## 04 — Feature Importance: ML Techniques

Quantifies which features carry predictive signal for `pitch_type` using correlation structure, Mutual Information, optional PCA experiments, and Random Forest importance across **tested candidate feature sets** (A–E).

**Sections:**
- Section 1 (exploratory): Symmetric outcome-block correlation and optional PCA for pitcher and batter — rejected for production
- Section 2: Random Forest MDI comparing candidate sets A–E

Run 02_data_quality and 03_pitch_distribution_eda first for context.

**Tested tiers** (from `feature_names.py`; RF feature counts reflect columns available after enrichment):
- A: game state only (11 defined → 11 RF features)
- B: game state + usage (31 defined → 29 RF features) — **chosen for training** (`MODEL_FEATURES`)
- C: B + pitcher outcome (171 defined → 169 RF features)
- D: B + batter outcome (171 defined → 141 RF features)
- E: B + both outcome blocks (311 defined → 281 RF features)

Production inputs are locked in code as **`MODEL_FEATURES`** (= Set B) in `feature_names.py`.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pybaseball
from pybaseball import cache, statcast

sys.path.append(str(Path("..").resolve()))
from utils.features.enrichment import enrich_statcast
from utils.features.feature_names import (
    BATTER_OUTCOME_COLUMNS,
    CANDIDATE_FEATURE_SETS,
    EDA_COLUMNS,
    LABEL_COLUMN,
    MODEL_FEATURES,
    PITCHER_OUTCOME_COLUMNS,
    PITCH_TYPES,
)
from utils.features.transforms import build_feature_matrix, feature_group, outcome_pca
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from matplotlib.patches import Patch

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("pybaseball :", pybaseball.__version__)
print("pandas     :", pd.__version__)
print("cache dir  :", cache.config.cache_directory)


In [ ]:
SEASON = 2024
PRIOR  = SEASON - 1

df = statcast("2024-06-01", "2024-06-30")
print(f"Raw shape: {df.shape}")

df_enr = enrich_statcast(df, PRIOR)
print(f"Enriched shape: {df_enr.shape}")
if "pitch_usage_FF" in df_enr.columns:
    print(f"Pitcher usage coverage (pitch_usage_FF non-null): {df_enr['pitch_usage_FF'].notna().mean():.1%}")

In [ ]:
available    = [c for c in EDA_COLUMNS if c in df_enr.columns]
missing_cols = [c for c in EDA_COLUMNS if c not in df_enr.columns]

model_df = df_enr[available + [LABEL_COLUMN]].copy()
model_df = model_df[model_df[LABEL_COLUMN].isin(PITCH_TYPES)].reset_index(drop=True)

print(f"model_df shape       : {model_df.shape}")
print(f"EDA_COLUMNS total    : {len(EDA_COLUMNS)}")
print(f"Found in enriched frame: {len(available)}")
if missing_cols:
    preview = missing_cols[:5]
    ellipsis = "..." if len(missing_cols) > 5 else ""
    print(f"Missing ({len(missing_cols)})           : {preview}{ellipsis}")
print(f"\nPitch type distribution:\n{model_df[LABEL_COLUMN].value_counts()}")

---
## Section 1 — Outcome Block Analysis (historical exploratory)

Historical exploratory — outcome blocks and PCA were **rejected** for production; kept for audit trail.

Symmetric analysis of pitcher and batter **outcome** columns (usage carved out). Determines whether compressing the 140-col blocks is worth testing in RF.

In [ ]:
bat_outcome_cols = [c for c in BATTER_OUTCOME_COLUMNS if c in model_df.columns]
pit_outcome_cols = [c for c in PITCHER_OUTCOME_COLUMNS if c in model_df.columns]
print(f"Batter outcome cols available : {len(bat_outcome_cols)}")
print(f"Pitcher outcome cols available: {len(pit_outcome_cols)}")

### Visual 1 — Batter outcome: 14×14 mean |r| across pitch types (stat families)
_OUTCOME_STAT_NAMES = sorted({c.rsplit("_", 1)[0] for c in bat_outcome_cols})

analysis_enr = df_enr[df_enr[LABEL_COLUMN].isin(PITCH_TYPES)]

def mean_abs_corr_block(cols, id_col, stat_names):
    ### Deduplicate to one row per player before correlating (MERGE_KEYS live on df_enr only)
    dedup = analysis_enr[[id_col] + cols].drop_duplicates(id_col)
    mat = dedup[cols].astype(float)
    n = len(stat_names)
    corr_blocks = np.zeros((n, n))
    for i, s1 in enumerate(stat_names):
        for j, s2 in enumerate(stat_names):
            c1 = [c for c in cols if c.startswith(s1 + "_")]
            c2 = [c for c in cols if c.startswith(s2 + "_")]
            sub = mat[c1 + c2].dropna(how="all")
            if sub.shape[1] < 2:
                continue
            r = sub.corr().abs()
            cross = r.loc[c1, c2].values.flatten()
            if len(cross):
                corr_blocks[i, j] = np.nanmean(cross)
    return corr_blocks

block = mean_abs_corr_block(bat_outcome_cols, "batter", _OUTCOME_STAT_NAMES)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    pd.DataFrame(block, index=_OUTCOME_STAT_NAMES, columns=_OUTCOME_STAT_NAMES),
    ax=ax, cmap="Blues", annot=True, fmt=".2f", annot_kws={"size": 7},
)
ax.set_title("Batter Outcome Stats — Mean |r| Across Pitch Types")
plt.tight_layout()
plt.show()

### Visual 2 — Pitcher outcome: 14×14 mean |r| across pitch types (stat families)
_pit_stat_names = sorted({c.rsplit("_", 1)[0] for c in pit_outcome_cols})
pit_block = mean_abs_corr_block(pit_outcome_cols, "pitcher", _pit_stat_names)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    pd.DataFrame(pit_block, index=_pit_stat_names, columns=_pit_stat_names),
    ax=ax, cmap="Oranges", annot=True, fmt=".2f", annot_kws={"size": 7},
)
ax.set_title("Pitcher Outcome Stats — Mean |r| Across Pitch Types")
plt.tight_layout()
plt.show()

In [ ]:
### Visual 3 — Optional PCA on batter outcome block (experiment)
MAX_PCS = 5

X_pcs_bat, pca_bat, pc_labels_bat, _, _ = outcome_pca(model_df, "batter", n_components=MAX_PCS)

if pca_bat.n_components_:
    loadings = pd.DataFrame(pca_bat.components_, index=pc_labels_bat, columns=bat_outcome_cols)
    top20 = loadings.abs().max(axis=0).nlargest(20).index
    fig, ax = plt.subplots(figsize=(16, 4))
    sns.heatmap(loadings[top20], ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 6})
    ax.set_title("Batter Outcome PCA Loadings (top 20 features)")
    plt.tight_layout()
    plt.show()
    cum = pca_bat.explained_variance_ratio_.cumsum()
    for i, v in enumerate(cum):
        print(f"  PC1–PC{i+1}: {v:.1%}")
else:
    print("No batter outcome cols for PCA")

### Visual 4 — Optional PCA on pitcher outcome block (experiment)
X_pcs_pit, pca_pit, pc_labels_pit, _, _ = outcome_pca(model_df, "pitcher", n_components=MAX_PCS)

if pca_pit.n_components_:
    loadings = pd.DataFrame(pca_pit.components_, index=pc_labels_pit, columns=pit_outcome_cols)
    top20 = loadings.abs().max(axis=0).nlargest(20).index
    fig, ax = plt.subplots(figsize=(16, 4))
    sns.heatmap(loadings[top20], ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 6})
    ax.set_title("Pitcher Outcome PCA Loadings (top 20 features)")
    plt.tight_layout()
    plt.show()
    cum = pca_pit.explained_variance_ratio_.cumsum()
    for i, v in enumerate(cum):
        print(f"  PC1–PC{i+1}: {v:.1%}")
else:
    print("No pitcher outcome cols for PCA")

In [ ]:
### Visual 5 — MI: batter PCs vs top raw outcome features
y = LabelEncoder().fit_transform(model_df[LABEL_COLUMN])

if X_pcs_bat.shape[1] > 0 and bat_outcome_cols:
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_raw = sc.fit_transform(imp.fit_transform(model_df[bat_outcome_cols]))

    mi_pcs = mutual_info_classif(X_pcs_bat, y, discrete_features=False, random_state=42)
    mi_raw = mutual_info_classif(X_raw, y, discrete_features=False, random_state=42)
    top10 = pd.Series(mi_raw, index=bat_outcome_cols).nlargest(10)
    best_raw = top10.max()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(mi_pcs)), mi_pcs, color="steelblue", label="PCA components")
    ax.bar(range(len(mi_pcs), len(mi_pcs) + len(top10)), top10.values, color="darkorange", label="Top raw cols")
    ax.axhline(best_raw, color="darkred", linestyle="--", label=f"Best raw ({best_raw:.4f})")
    ax.legend()
    ax.set_title("Batter outcome: PC MI vs top-10 raw MI (exploratory)")
    plt.tight_layout()
    plt.show()
    print("PCA is optional — compare candidate sets in Section 2 before committing.")

In [ ]:
### Visual 6 — MI: pitcher PCs vs top raw outcome features
if X_pcs_pit.shape[1] > 0 and pit_outcome_cols:
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_raw_pit = sc.fit_transform(imp.fit_transform(model_df[pit_outcome_cols]))

    mi_pcs_pit = mutual_info_classif(X_pcs_pit, y, discrete_features=False, random_state=42)
    mi_raw_pit = mutual_info_classif(X_raw_pit, y, discrete_features=False, random_state=42)
    top10_pit = pd.Series(mi_raw_pit, index=pit_outcome_cols).nlargest(10)
    best_raw_pit = top10_pit.max()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(mi_pcs_pit)), mi_pcs_pit, color="darkorange", label="PCA components")
    ax.bar(
        range(len(mi_pcs_pit), len(mi_pcs_pit) + len(top10_pit)),
        top10_pit.values,
        color="steelblue",
        label="Top raw cols",
    )
    ax.axhline(best_raw_pit, color="darkred", linestyle="--", label=f"Best raw ({best_raw_pit:.4f})")
    ax.legend()
    ax.set_title("Pitcher outcome: PC MI vs top-10 raw MI (exploratory)")
    plt.tight_layout()
    plt.show()
    print("Outcome blocks are highly correlated — PCA compression is optional, not default.")

---
## Section 2 — Random Forest Feature Importance (candidate sets A–E)

Compare OOB accuracy and MDI importance across candidate column sets using `build_feature_matrix()`.

In [ ]:
### Build matrices and train RF per candidate set (experiment registry — production uses MODEL_FEATURES)
CANDIDATE_SET_LABELS = {
    "A": "A — game state",
    "B": "B — + usage",
    "C": "C — + pitcher outcome",
    "D": "D — + batter outcome",
    "E": "E — + both outcomes",
}

y = LabelEncoder().fit_transform(model_df[LABEL_COLUMN])
results = []

for key in "ABCDE":
    label = CANDIDATE_SET_LABELS[key]
    cols = CANDIDATE_FEATURE_SETS[key]
    X, names, _ = build_feature_matrix(model_df, cols)
    rf = RandomForestClassifier(
        n_estimators=200, class_weight="balanced", oob_score=True,
        random_state=42, n_jobs=-1,
    )
    rf.fit(X, y)
    results.append({"set": label, "n_features": X.shape[1], "oob_accuracy": rf.oob_score_})
    print(f"{label}: {X.shape[1]} features, OOB acc = {rf.oob_score_:.3f}")

pd.DataFrame(results)

In [ ]:
### RF on MODEL_FEATURES (Set B) — top 30 importances
X, feature_names_out, _ = build_feature_matrix(model_df, MODEL_FEATURES)
rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", oob_score=True,
    random_state=42, n_jobs=-1,
)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_names_out)
top30 = importances.nlargest(30).sort_values()
groups = [feature_group(f) for f in top30.index]

GROUP_COLORS = {
    "Game state": "seagreen",
    "Pitcher usage": "darkorange",
    "Batter usage": "mediumpurple",
    "Pitcher outcome": "firebrick",
    "Batter outcome": "steelblue",
    "pitcher outcome PCA": "steelblue",
    "batter outcome PCA": "steelblue",
    "Other": "gray",
}

bar_colors = [GROUP_COLORS.get(g, "gray") for g in groups]
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top30.index, top30.values, color=bar_colors, edgecolor="white", height=0.72)
ax.set_xlabel("Mean Decrease in Impurity (MDI)")
ax.set_title(f"MODEL_FEATURES (Set B) — Top 30 Importances (OOB acc: {rf.oob_score_:.3f})")
legend_patches = [Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()

group_imp = (
    pd.DataFrame({"feature": feature_names_out, "importance": rf.feature_importances_})
    .assign(group=lambda d: d["feature"].map(feature_group))
    .groupby("group")["importance"].sum().sort_values(ascending=False)
)
print("\nTotal MDI by group (MODEL_FEATURES):")
for grp, val in group_imp.items():
    print(f"  {grp:<22} {val:.4f}")

---
## Findings / recommendations

**Locked in code:** `MODEL_FEATURES` in `feature_names.py` (= candidate Set B).

June 2024 Statcast sample (114k pitches, prior-year enrichment from 2023):

1. **Set B (game state + usage) is good**. OOB accuracy ~0.39 vs ~0.18 for game state alone under a simple unoptimized random forest.  

2. Pitch usage and certain game state information (balls, strikes, run differential) have high RF feature importance. 

3. Raw outcome blocks do not beat set B. Sets C/D/E plateau around 0.37–0.38 OOB — slightly below set B despite 5–10× more features (169 / 141 / 281 RF features vs 29 for set B). This means most outcome statistics simply are not very informative. 

4. PCA on pitcher/batter outcomes are not super useful. Five components explain ~44% of batter outcome variance and similarly for pitcher. Using PCA on outcomes also do not improve OOB performance relative to not using all of them, or not using outcomes (candidate B) at all. 

5. Rare pitch types lack prior-year arsenal rows. Usage imputation (zero when any usage present, global prior otherwise) is handled in `build_feature_matrix(..., MODEL_FEATURES)`.

Train on **`MODEL_FEATURES`** — game state plus pitcher and batter usage priors (29 encoded features).